In [1]:
!pip install bert-score rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=54649b5822b40a2ee6741ca6eb6eb16c7f92efc187f3b9fc0abca1d011cce47a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import time

from bert_score import BERTScorer
from rouge_score import rouge_scorer

r_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
std_scorer = BERTScorer(model_type="bert-base-uncased", lang="en", device="cpu")
code_scorer = BERTScorer(model_type="microsoft/codebert-base", num_layers=12, lang="en", device="cpu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
refs = ["Use increment operator instead of x = x + 1"]
cands = ["x += 1 is preferred over x = x + 1"]

In [5]:
def run_benchmark(name, func, iterations=100):
    print(f"Running benchmark for {name}...")
    _ = func()

    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        _ = func()
        times.append((time.perf_counter() - start) * 1000)

    avg_time = sum(times) / len(times)
    return avg_time


def rouge_fn():
    scores = r_scorer.score(refs[0], cands[0])
    return scores['rougeL'].fmeasure


def bert_fn():
    P, R, F1 = std_scorer.score(cands, refs)
    return F1.mean().item()


def codebert_fn():
    P, R, F1 = code_scorer.score(cands, refs)
    return F1.mean().item()


results = {}
results['ROUGE-L'] = (run_benchmark("ROUGE-L", rouge_fn), rouge_fn())
results['BERTScore'] = (run_benchmark("BERTScore", bert_fn), bert_fn())
results['CodeBERTScore'] = (run_benchmark("CodeBERTScore", codebert_fn), codebert_fn())

print(f"\n{'Metric':<20} | {'Avg Time (ms)':<15} | {'Score':<10}")
for name, (t, s) in results.items():
    print(f"{name:<20} | {t:>13.2f} ms | {s:>8.4f}")

Running benchmark for ROUGE-L...
Running benchmark for BERTScore...
Running benchmark for CodeBERTScore...

Metric               | Avg Time (ms)   | Score     
ROUGE-L              |          0.21 ms |   0.3750
BERTScore            |        143.27 ms |   0.7038
CodeBERTScore        |        123.10 ms |   0.9194
